# PB Smear 백혈구 분류 모델 학습 (EfficientNetB0)
런타임 > 런타임 유형 변경 > **T4 GPU** 선택 후 **Step 1부터 순서대로 전체 실행**하세요.

In [ ]:
# =============================================
# [Step 1] 구글 드라이브 마운트
# =============================================
from google.colab import drive
drive.mount('/content/drive')

DATASET_DIR     = '/content/drive/MyDrive/PB_Smear/dataset'
MODEL_SAVE_PATH = '/content/drive/MyDrive/PB_Smear/model.h5'
MERGED_DIR      = '/content/merged_dataset'
print('드라이브 마운트 완료!')

In [ ]:
# =============================================
# [Step 2] 라이브러리 임포트
# =============================================
import os, glob, shutil, json, random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input  # ★ EfficientNet 전용 전처리
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator, load_img, img_to_array, array_to_img
)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print(f'TensorFlow 버전: {tf.__version__}')
print(f'GPU 사용 가능: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# =============================================
# [Step 3] 하이퍼파라미터 설정
# =============================================
IMG_SIZE        = 224
BATCH_SIZE      = 32
EPOCHS_FROZEN   = 15
EPOCHS_FINETUNE = 30
LEARNING_RATE   = 1e-3
NUM_CLASSES     = 5

CLASS_NAMES = ['Basophil', 'Eosinophil', 'Lymphocyte', 'Monocyte', 'Neutrophil']
ALL_LABELS  = list(range(NUM_CLASSES))

In [ ]:
# =============================================
# [Step 4] Kaggle 인증 & 데이터셋 다운로드
#
# 데이터셋: paultimothymooney/blood-cells
#   포함 클래스: EOSINOPHIL / LYMPHOCYTE / MONOCYTE / NEUTROPHIL (각 ~3000장)
#   Basophil 없음 → Step 6 에서 기존 42장 증강으로 보완
#
# ★ KAGGLE_KEY 에 본인 API 토큰을 붙여넣으세요.
# =============================================
KAGGLE_USERNAME = 'jaeduk'
KAGGLE_KEY      = 'KGAT_c2bdbf24b8b6f4b290215bb7fa364c82'   # ← Kaggle API 토큰

!pip install kaggle -q
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('kaggle.json 생성 완료')

!kaggle datasets download -d paultimothymooney/blood-cells -p /content/kaggle_data --unzip -q
print('Kaggle 다운로드 완료!')

# 폴더 구조 확인
for root, dirs, files_list in os.walk('/content/kaggle_data'):
    level = root.replace('/content/kaggle_data', '').count(os.sep)
    if level < 4:
        n = len([f for f in files_list if f.lower().endswith(('.jpg','.jpeg','.png'))])
        suffix = f'  ({n}장)' if n > 0 else ''
        print('  ' * level + os.path.basename(root) + '/' + suffix)

In [ ]:
# =============================================
# [Step 5] 기존 데이터 + Kaggle 데이터 병합
# =============================================
if os.path.exists(MERGED_DIR):
    shutil.rmtree(MERGED_DIR)
for name in CLASS_NAMES:
    os.makedirs(os.path.join(MERGED_DIR, name), exist_ok=True)

# 기존 드라이브 데이터 복사
copied_existing = 0
for name in CLASS_NAMES:
    for img_path in (glob.glob(os.path.join(DATASET_DIR, name, '*.jpg')) +
                     glob.glob(os.path.join(DATASET_DIR, name, '*.jpeg')) +
                     glob.glob(os.path.join(DATASET_DIR, name, '*.png'))):
        shutil.copy(img_path, os.path.join(MERGED_DIR, name, f'orig_{os.path.basename(img_path)}'))
        copied_existing += 1
print(f'기존 데이터 복사: {copied_existing}장')

# Kaggle 데이터 복사 (대소문자 무관 매핑)
kaggle_map = {
    'EOSINOPHIL': 'Eosinophil', 'LYMPHOCYTE': 'Lymphocyte',
    'MONOCYTE':   'Monocyte',   'NEUTROPHIL': 'Neutrophil',
    'BASOPHIL':   'Basophil',
}
MAX_PER_CLASS = 1500
copied_kaggle = {name: 0 for name in CLASS_NAMES}

for root, dirs, files_list in os.walk('/content/kaggle_data'):
    folder = os.path.basename(root).upper()
    if folder in kaggle_map:
        target = kaggle_map[folder]
        imgs   = [f for f in files_list if f.lower().endswith(('.jpg','.jpeg','.png'))]
        for i, img_file in enumerate(imgs[:MAX_PER_CLASS]):
            shutil.copy(
                os.path.join(root, img_file),
                os.path.join(MERGED_DIR, target, f'kag_{i:04d}_{img_file}')
            )
            copied_kaggle[target] += 1

print('\nKaggle 데이터 병합 결과:')
print(f'  {"클래스":12s} {"기존":>6s} {"Kaggle":>8s} {"합계":>6s}')
print('  ' + '-'*36)
for name in CLASS_NAMES:
    orig  = len(glob.glob(os.path.join(DATASET_DIR, name, '*')))
    kag   = copied_kaggle[name]
    total = len(glob.glob(os.path.join(MERGED_DIR, name, '*')))
    print(f'  {name:12s} {orig:6d} {kag:8d} {total:6d}')

In [ ]:
# =============================================
# [Step 6] Basophil 증강 — 42장 → 150장
# Kaggle 데이터에 Basophil 없으므로 기존 이미지 증강으로 보완
# =============================================
TARGET_BASOPHIL = 150
baso_dst     = os.path.join(MERGED_DIR, 'Basophil')
current_baso = (glob.glob(os.path.join(baso_dst, '*.jpg')) +
                glob.glob(os.path.join(baso_dst, '*.jpeg')) +
                glob.glob(os.path.join(baso_dst, '*.png')))

aug_gen = ImageDataGenerator(
    rotation_range=45, width_shift_range=0.15, height_shift_range=0.15,
    zoom_range=0.25, horizontal_flip=True, vertical_flip=True,
    brightness_range=[0.7, 1.3], fill_mode='nearest'
)
needed = max(0, TARGET_BASOPHIL - len(current_baso))
random.shuffle(current_baso)

for i in range(needed):
    src  = current_baso[i % len(current_baso)]
    img  = np.expand_dims(img_to_array(load_img(src, target_size=(IMG_SIZE, IMG_SIZE))), 0)
    aug  = next(aug_gen.flow(img, batch_size=1))[0].astype('uint8')
    array_to_img(aug).save(os.path.join(baso_dst, f'aug_{i:04d}.jpg'))

print(f'Basophil: {len(current_baso)}장 → {len(glob.glob(os.path.join(baso_dst, "*")))}장')

In [ ]:
# =============================================
# [Step 7] 병합 데이터셋 최종 확인
# =============================================
print('=== 병합 데이터셋 클래스별 이미지 수 ===')
total = 0
for name in CLASS_NAMES:
    imgs  = (glob.glob(os.path.join(MERGED_DIR, name, '*.jpg')) +
             glob.glob(os.path.join(MERGED_DIR, name, '*.jpeg')) +
             glob.glob(os.path.join(MERGED_DIR, name, '*.png')))
    count = len(imgs)
    total += count
    print(f'  {name:12s}: {count:5d}개  {"█" * (count // 50)}')
print(f'  {"합계":12s}: {total:5d}개')

In [ ]:
# =============================================
# [Step 8] 데이터 로더 설정
#
# ★ 핵심: rescale=1./255 를 사용하지 않는다.
#   EfficientNetB0 의 ImageNet 사전학습 가중치는
#   preprocess_input 으로 정규화된 입력을 기대한다.
#   1./255 스케일링은 입력 분포를 왜곡시켜 Mode Collapse 를 유발했음.
# =============================================
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # ← EfficientNet 전용 정규화
    validation_split=0.2,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # ← 검증도 동일 전처리
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    MERGED_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', shuffle=True
)
val_generator = val_datagen.flow_from_directory(
    MERGED_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', shuffle=False
)
print(f'학습: {train_generator.samples}장 / 검증: {val_generator.samples}장')
print(f'클래스 인덱스: {train_generator.class_indices}')

In [ ]:
# =============================================
# [Step 9] 클래스 불균형 보정 — sklearn 자동 계산
#
# compute_class_weight('balanced') 공식:
#   weight = total_samples / (n_classes × class_samples)
#
# Basophil(~120장) vs 나머지(~1200장) → 원시 가중치 ~8.0
# 상한 MAX_WEIGHT=3.0: Basophil 과도한 가중치가 역방향 collapse 유발 방지
# =============================================
MAX_WEIGHT = 3.0

labels        = train_generator.classes
unique_labels = np.unique(labels)
computed_w    = compute_class_weight('balanced', classes=unique_labels, y=labels)

class_weight_dict = {i: 1.0 for i in range(NUM_CLASSES)}  # 누락 클래스 기본값 1.0
for lbl, w in zip(unique_labels, computed_w):
    class_weight_dict[lbl] = min(w, MAX_WEIGHT)

print(f'클래스별 가중치 (상한 {MAX_WEIGHT}):')
print(f'  {"클래스":12s} {"샘플수":>8s} {"원시가중치":>12s} {"적용가중치":>12s}')
print('  ' + '-'*48)
for lbl, w in zip(unique_labels, computed_w):
    count = int(np.sum(labels == lbl))
    applied = class_weight_dict[lbl]
    capped  = ' ← 상한 적용' if w > MAX_WEIGHT else ''
    print(f'  {CLASS_NAMES[lbl]:12s} {count:8d} {w:12.3f} {applied:12.3f}{capped}')

In [ ]:
# =============================================
# [Step 10] EfficientNetB0 모델 구축
#
# Head 구조 (튼튼한 분류기):
#   GlobalAveragePooling2D
#     → BatchNormalization  (내부 공변량 이동 방지)
#     → Dense(256, relu)
#     → Dropout(0.5)        (과적합 강하게 억제)
#     → Dense(5, softmax)
# =============================================
base_model = EfficientNetB0(include_top=False, weights='imagenet',
                             input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = base_model(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dense(256, activation='relu')(x)
x       = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs, outputs)
model.summary()

In [ ]:
# =============================================
# [Step 11] 1단계 학습 — 헤드만 학습 (backbone 동결)
#
# 콜백:
#   ReduceLROnPlateau: val_loss 2 에포크 개선 없으면 lr × 0.5
#   EarlyStopping    : val_loss 5 에포크 개선 없으면 조기 종료
# =============================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
history1 = model.fit(
    train_generator,
    epochs=EPOCHS_FROZEN,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=[
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2,
                          min_lr=1e-5, verbose=1),
        EarlyStopping(monitor='val_loss', patience=5,
                      restore_best_weights=True, verbose=1),
    ],
    verbose=1
)
print('1단계 완료!')

In [ ]:
# =============================================
# [Step 12] 2단계 학습 — Fine-tuning (마지막 30 레이어 해동)
#
# 콜백:
#   ModelCheckpoint  : val_accuracy 최고일 때 model.h5 저장
#   ReduceLROnPlateau: val_loss 2 에포크 개선 없으면 lr × 0.5
#   EarlyStopping    : val_loss 5 에포크 개선 없으면 조기 종료
# =============================================
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE / 10),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
history2 = model.fit(
    train_generator,
    epochs=EPOCHS_FINETUNE,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=[
        ModelCheckpoint(MODEL_SAVE_PATH, save_best_only=True,
                        monitor='val_accuracy', verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2,
                          min_lr=1e-5, verbose=1),
        EarlyStopping(monitor='val_loss', patience=5,
                      restore_best_weights=True, verbose=1),
    ],
    verbose=1
)
print(f'Fine-tuning 완료! 저장 위치: {MODEL_SAVE_PATH}')

In [ ]:
# =============================================
# [Step 13] 학습 곡선 시각화
# =============================================
split = len(history1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (tk, vk), title in zip(axes,
    [('accuracy','val_accuracy'), ('loss','val_loss')],
    ['Accuracy', 'Loss']):
    tr = history1.history[tk]  + history2.history[tk]
    vl = history1.history[vk] + history2.history[vk]
    ax.plot(tr, label='Train'); ax.plot(vl, label='Val')
    ax.axvline(x=split, color='r', linestyle='--', label='Fine-tune 시작')
    ax.set_title(title); ax.legend()
plt.suptitle('EfficientNetB0 학습 결과', fontsize=14)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/PB_Smear/training_history.png')
plt.show()

In [ ]:
# =============================================
# [Step 14] 클래스별 예측 진단 + 혼동 행렬
#
# ★ model.h5 를 직접 로드해서 평가:
#    model.fit() 종료 후 메모리 상 모델은 EarlyStopping(val_loss 기준) 복원본.
#    디스크의 model.h5 는 ModelCheckpoint(val_accuracy 기준) 저장본.
#    앱에서 실제 사용하는 model.h5 의 성능을 정확히 측정한다.
# =============================================
from tensorflow.keras.models import load_model as _load_model

eval_model = _load_model(MODEL_SAVE_PATH)
print(f'평가 모델 로드 완료: {MODEL_SAVE_PATH}')

val_generator.reset()
preds       = eval_model.predict(val_generator, verbose=1)
true_labels = val_generator.classes

# ★ 배치 패딩으로 예측이 실제 샘플보다 많을 수 있으므로 잘라냄
pred_labels = np.argmax(preds, axis=1)[:len(true_labels)]

print(f'\n예측 샘플 수: {len(pred_labels)} / 실제 샘플 수: {len(true_labels)}')

print('\n=== 클래스별 예측 분포 ===')
for i, name in enumerate(CLASS_NAMES):
    count = int(np.sum(pred_labels == i))
    print(f'  {name:12s}: {count:4d}개  {"█" * min(count // 5, 50)}')

print('\n=== Classification Report ===')
print(classification_report(true_labels, pred_labels,
      labels=ALL_LABELS, target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(true_labels, pred_labels, labels=ALL_LABELS)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix'); plt.ylabel('실제'); plt.xlabel('예측')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/PB_Smear/confusion_matrix.png')
plt.show()

n = len(np.unique(pred_labels))
if   n == 1: print(f'\n⚠ 모든 예측이 {CLASS_NAMES[pred_labels[0]]}로 몰렸습니다.')
elif n <  5: print(f'\n⚠ {n}개 클래스만 예측됩니다.')
else:        print('\n✓ 5개 클래스 모두 예측됩니다. model.h5 를 로컬에 적용하세요!')